# Example 10.2 / 2: Solve an Assignment Problem

Companion notebook for **10.2 Transportation and Assignment Networks** in the AMPL lab Documentation, section "Solve an Assignment Problem". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Solve an Assignment Problem

Set every supply and demand to one. Under the conditions in Total Unimodularity and Integral Flows (AMPL lab documentation, section 10.1.2), a continuous `Assign` variable has a binary-valued optimum. Use explicit binary variables when additional rules break that structure.

Three workers each take exactly one of three jobs, and each job gets exactly one worker:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set WORKERS;                                 # people to assign
set JOBS;                                    # tasks to cover
param cost {WORKERS, JOBS} >= 0;             # cost of giving job j to worker w

var Assign {WORKERS, JOBS} >= 0;             # continuous; the optimum is 0 or 1

minimize TotalCost:
    sum {w in WORKERS, j in JOBS} cost[w, j] * Assign[w, j];

subject to OneJobPerWorker {w in WORKERS}:   # each worker takes one job
    sum {j in JOBS} Assign[w, j] = 1;

subject to OneWorkerPerJob {j in JOBS}:      # each job gets one worker
    sum {w in WORKERS} Assign[w, j] = 1;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["WORKERS"] = ["Ana", "Bo", "Cy"]
ampl.set["JOBS"] = ["cut", "pack", "ship"]
ampl.param["cost"] = {("Ana", "cut"): 2, ("Ana", "pack"): 5, ("Ana", "ship"): 4,
                      ("Bo", "cut"): 3, ("Bo", "pack"): 1, ("Bo", "ship"): 3,
                      ("Cy", "cut"): 4, ("Cy", "pack"): 2, ("Cy", "ship"): 1}

In [5]:
# run.py
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result, ampl.obj["TotalCost"].value())
assign = ampl.var["Assign"].to_dict()
print({w: j for (w, j), value in assign.items() if value > 0.5})

solved 4.0
{'Ana': 'cut', 'Bo': 'pack', 'Cy': 'ship'}


### Pure AMPL: data.dat and model.run

Every `Assign` value is 0 or 1 although the variable is continuous, so the pairing can be read
directly from the values equal to 1.

Pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set WORKERS := Ana Bo Cy;
set JOBS := cut pack ship;
param cost:  cut pack ship :=
  Ana         2    5    4
  Bo          3    1    3
  Cy          4    2    1 ;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
option solver_msg 0;
solve;
display TotalCost;
display {w in WORKERS, j in JOBS: Assign[w, j] > 0.5} Assign[w, j];

TotalCost = 4

Assign[w,j] :=
Ana cut    1
Bo  pack   1
Cy  ship   1
;

